Store complete interaction episodes with temporal context, so the agent can recall "what happened when" across sessions.

You don't remember your life as a flat list of facts. You remember episodes: bounded experiences tied to a time and place. "The meeting where we decided to pivot the product." "The debugging session that took all afternoon." Each episode has a beginning, a middle, and an end.

Episodic Memory brings this capability to AI agents. It captures entire conversation sessions (or meaningful segments) as discrete episodes. Each episode is tagged with timestamps, topic labels, and an auto-generated summary. This lets the agent recall past experiences as coherent events, not isolated facts.

This matters because many agent use cases span multiple sessions. A coaching agent needs to recall what goals were set last week. A project-management agent needs to know what was decided three days ago. Without episodic memory, each session starts from scratch. The agent has amnesia between interactions.

The key engineering challenge is episode boundary detection (deciding where one episode ends and another begins). Boundaries can be session-based (one session = one episode), topic-based (a topic shift starts a new episode), or time-based (a gap of n minutes triggers a new episode). A second challenge is retrieval: given a new user query, how do you find the most relevant past episodes? This requires a mix of temporal indexing, semantic similarity, and explicit reference detection.

By the end of this notebook you'll understand:

How to capture conversation turns into structured episodes with metadata.

How to detect episode boundaries (session-based and topic-based).

How to generate episode summaries with an LLM.

How to retrieve relevant episodes using recency and semantic similarity.

When episodic memory helps and when it quietly fails.

### Key Concepts

Episode: A bounded segment of conversation stored as a single unit. Think of it as one chapter in a journal. It has a start time, an end time, a list of messages, and a summary.

Episode boundary: The dividing line between two episodes. Boundaries can be session-based (one session = one episode), topic-based (a detected topic shift starts a new episode), or time-based (a gap of n minutes triggers a new episode).

Temporal indexing: Tagging each episode with timestamps (start time, end time, duration). This enables queries like "what did we discuss last Tuesday?" or "what happened in the previous session?"

Episode summary: A short text that captures the gist of an episode. The LLM (large language model, the AI that generates text) produces this summary when the episode closes. Summaries let you scan past episodes quickly without loading full transcripts.

Embedding: A list of numbers that represents the meaning of a piece of text. Two texts with similar meaning have embeddings that are close together in vector space. We use embeddings to find episodes that are semantically related to a query.

Cosine similarity: A way to measure how close two embeddings are. It ranges from -1 (opposite) to 1 (identical). Higher values mean the texts are more related.

Episodic vs. semantic memory: Episodic memory stores specific experiences ("the user asked me to rewrite the intro on March 5th"). Semantic memory stores generalized knowledge ("the user prefers concise writing"). Both are valuable but serve different roles.

Episode decay: Older episodes may be compressed or pruned to keep storage manageable in long-lived agents.

> **The memory type family so far:**
>
> | Type | Stores | Answers | Technique |
> |---|---|---|---|
> | Short-term | Recent conversation turns | What are we talking about right now? | 01–05 |
> | Vector store | Raw message text as embeddings | What did we discuss about X? | 06 |
> | Entity | Structured current facts | What is Chiru's salary right now? | 07 |
> | **Episodic** | **Complete timestamped interaction records** | **What happened on June 3rd? What did we decide in the April session?** | **09** |
>
> Episodic memory is the agent's **diary** — a structured, timestamped, immutable log of everything that happened, session by session.

---

## What Is Episodic Memory?

### The core idea in one sentence
Store every session as a complete, timestamped **episode** — a structured record capturing what was discussed, what decisions were made, what advice was given, and what the emotional context was — and retrieve relevant episodes when they are needed in future sessions.

---

### The cognitive science foundation

Endel Tulving (1972) distinguished two types of long-term memory:
- **Episodic memory**: memory of specific events in time — *what happened, when, and where*
- **Semantic memory**: general knowledge independent of when it was learned

When you remember "I had a difficult conversation with my financial advisor on March 14th about my portfolio losses" — that is episodic. When you know "mutual funds are diversified investment vehicles" — that is semantic.

AI agents need both. Technique 07 (Entity Memory) and Technique 10 (Semantic Memory) handle the semantic side. Technique 09 handles the episodic side.

---

### The mental model — a structured diary

Every session produces one episode entry:

```
EPISODE: session_vs_001
────────────────────────────────────────────────────────────
date:          2025-06-12
duration:      18 minutes
topics:        [salary, expenses, FD, SIP, retirement planning]
decisions:     User decided to start a ₹5,000/month SIP in debt funds
advice_given:  Recommended HDFC Short Duration Fund for FD maturity proceeds
user_state:    Anxious about market volatility, conservative stance reinforced
key_numbers:   {salary: ₹1,20,000, surplus: ₹60,000, FD: ₹50,000}
outcome:       Positive — user committed to action plan
────────────────────────────────────────────────────────────
```

When Chiru returns three months later and asks "what did we decide last time about my FD?", the agent retrieves this episode and answers precisely.

---

### How it differs from previous techniques

| | Vector Store (06) | Entity Memory (07) | Episodic Memory (09) |
|---|---|---|---|
| Unit stored | Individual messages | Individual facts | Complete sessions |
| Granularity | Message-level | Field-level | Session-level |
| Time awareness | Timestamp as metadata | Last-updated timestamp | Episode date, duration, sequence |
| Retrieval | Semantic search | Direct lookup | Semantic + temporal + type filters |
| Answers | "Find messages about X" | "What is X right now" | "What happened in session Y" |
| Immutability | Append-only | Updated in-place | **Immutable** — episodes never change |

The immutability is critical: episodic memory is an audit trail. You cannot rewrite history. Every episode is a permanent record of what the agent knew and said at a specific point in time.

---

### Point 1 — Episodes are generated at session end, not turn by turn

Unlike vector store memory (which embeds every message as it arrives), episodic memory is synthesised at session end. After the session closes, we run a structured summarisation that produces the episode record. This approach:
- Captures session-level patterns that individual turns cannot
- Produces a rich, dense summary rather than scattered message chunks
- Runs asynchronously — the user never waits for it

---

### Point 2 — Episode retrieval uses multiple strategies

Unlike entity memory (direct lookup) or vector store (semantic only), episodic memory benefits from multiple retrieval strategies:
- **Semantic**: find episodes where topics similar to the current query were discussed
- **Temporal**: find the most recent N episodes, or episodes from a specific date range
- **Type-based**: find episodes where a specific type of event occurred (e.g. "all sessions where a decision was made")
- **Hybrid**: combine all three with weighted scoring

---

### Point 3 — Episodic memory is your compliance and audit layer

In regulated financial services, you must be able to answer: "What advice did your agent give this client on April 3rd and based on what information?" Vector stores give you message-level logs. Entity memory gives you a current-state profile. Neither gives you a session-level narrative of what happened and why.

Episodic memory does. It is simultaneously a UX feature (the agent remembers your history) and a compliance requirement (the firm can audit advice given).

---

### Point 4 — Case-based reasoning: learning from past episodes

When a similar situation arises, episodic memory enables the agent to reason from precedent: "Last time Chiru had a large FD maturity, he was anxious about market risk and preferred debt funds. The same situation has recurred. Apply the same approach unless contradicted."

This is case-based reasoning — one of the most powerful capabilities episodic memory enables. It is what makes an agent feel experienced rather than perpetually new.

---

## Trade-offs

| | |
|---|---|
| ✅ Session-level narrative context | ❌ Episode generation adds post-session latency |
| ✅ Compliance and audit trail | ❌ Episode quality depends on generation prompt |
| ✅ Case-based reasoning from past sessions | ❌ More expensive than raw message storage |
| ✅ Temporal reasoning ("what happened in April") | ❌ Does not replace entity or vector memory |
| ✅ Immutable — history cannot be rewritten | ❌ Episode retrieval adds tokens per call |

## Production Verdict

> **Essential for regulated domains and long-running agent relationships. Not a replacement for entity or vector memory — a complement.**
>
> In financial services, episodic memory is simultaneously a product feature ("FinCoach remembers every session we've had") and a regulatory requirement (audit trail of advice given). Build it if your domain has compliance requirements, long-running user relationships, or explicit user expectations of session-level recall.

---

In [1]:
# Install required packages.
# 'openai'      — GPT-4o for conversation + episode generation + embedding.
# 'chromadb'    — vector store for episode retrieval by semantic similarity.
# 'tiktoken'    — exact token counting for GPT-4o.
!pip install openai chromadb tiktoken --quiet

In [2]:
# --- Standard library ---
import json                              # Serialisation.
import os                                # Environment variables.
import time                              # Rate-limit delays.
import uuid                              # Unique episode IDs.
from datetime import datetime, timezone, timedelta  # Timestamps and date math.
from typing import List, Dict, Optional, Any  # Type hints.
from dataclasses import dataclass, field, asdict  # Data models.
from enum import Enum                    # Episode outcome enum.

# --- Third-party ---
import openai    # OpenAI SDK.
import chromadb  # Vector database for episode retrieval.
import tiktoken  # Exact GPT-4o tokeniser.

In [3]:
# --- API Key Setup ---
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid."

client = openai.OpenAI(api_key=OPENAI_API_KEY)

CHAT_MODEL      = "gpt-4o"
# Main conversation model.

EPISODE_MODEL   = "gpt-4o-mini"
# Used for episode generation — structured summarisation task.
# 16x cheaper than gpt-4o, sufficient quality for summarisation.

EMBEDDING_MODEL = "text-embedding-3-small"
# Used for episode retrieval — embeds episode summaries for semantic search.

TOKENISER = tiktoken.get_encoding("o200k_base")

# --- ChromaDB setup for episode store ---
CHROMA_PERSIST_DIR  = "/tmp/fincoach_episodes_chromadb"
# Separate from Technique 06's message-level store.
# Episodes and individual messages are different granularities — separate collections.

chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
EPISODE_COLLECTION  = "fincoach_episodes"
# Collection name — one episode per session, stored here.

print(f"Chat model     : {CHAT_MODEL}")
print(f"Episode model  : {EPISODE_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Key loaded from environment variable.
Chat model     : gpt-4o
Episode model  : gpt-4o-mini
Embedding model: text-embedding-3-small


---
## Part 1 — The Episode Data Model

An episode is a structured record of a complete session. Every field is designed to answer a specific type of recall question.

In [4]:
class EpisodeOutcome(str, Enum):
    """
    The outcome of a session from the agent's perspective.
    Used for filtering and case-based reasoning.
    """
    POSITIVE    = "positive"    # User engaged, decisions made, positive sentiment.
    NEUTRAL     = "neutral"     # Informational session, no specific decisions.
    UNRESOLVED  = "unresolved"  # User had concerns that were not fully addressed.
    NEGATIVE    = "negative"    # User was frustrated, confused, or dissatisfied.


@dataclass
class Episode:
    """
    A complete, immutable record of one conversation session.

    Every field serves a specific retrieval or audit purpose:
    - episode_id, session_id: identification and linking
    - user_id: multi-tenant isolation
    - date, turn_count, duration_minutes: temporal and usage context
    - topics, decisions, advice_given: the substance of what happened
    - key_numbers: the specific financial figures discussed
    - user_state, concerns: the emotional and contextual dimension
    - outcome: coarse-grained quality signal for case-based reasoning
    - episode_summary: the rich text used for semantic retrieval
    """

    episode_id:       str
    # Unique ID for this episode — used as the ChromaDB document ID.
    # Generated as a UUID4 when the episode is created.

    session_id:       str
    # The session that produced this episode — links back to the conversation logs.

    user_id:          str
    # Multi-tenant isolation — every ChromaDB query filters by this.

    date:             str
    # ISO date of the session: "2025-06-12"
    # Used for temporal filtering: "find all sessions from Q2 2025".

    timestamp:        str
    # Full UTC ISO timestamp: "2025-06-12T10:30:00Z"
    # Used for exact ordering and precise temporal queries.

    turn_count:       int
    # Number of conversation turns in this session.
    # Correlates with session depth — longer sessions = more context.

    duration_minutes: Optional[float]
    # How long the session lasted (if calculable from timestamps).

    topics:           List[str]
    # Main topics discussed: ["salary", "SIP", "retirement", "FD maturity"]
    # Used for type-based filtering: "find all sessions where SIPs were discussed".

    decisions:        List[str]
    # Specific decisions the user committed to:
    # ["Start ₹5,000/month SIP", "Renew FD for 1 year"]
    # Critical for compliance: what did the user agree to do?

    advice_given:     List[str]
    # Key recommendations the agent made:
    # ["Recommended HDFC Short Duration Fund for FD proceeds"]
    # Critical for compliance: what did the agent recommend?

    key_numbers:      Dict[str, str]
    # Financial figures mentioned:
    # {"salary": "₹1,20,000", "fd_amount": "₹50,000", "sip_target": "₹5,000"}
    # Enables precise numerical recall across sessions.

    user_state:       str
    # Brief description of user's emotional/situational context:
    # "anxious about market volatility, conservative stance"
    # Enables emotional continuity — the agent remembers how the user was feeling.

    concerns:         List[str]
    # Issues the user raised that were not fully resolved:
    # ["Uncertain about which debt fund to choose"]
    # Enables follow-up: "Last time you were unsure about debt funds — have you decided?"

    outcome:          str
    # One of the EpisodeOutcome values.
    # Used for case-based reasoning — prefer strategies from 'positive' episodes.

    episode_summary:  str
    # A rich natural language summary of the episode.
    # THIS IS WHAT GETS EMBEDDED for semantic retrieval.
    # Contains all the above in prose form — dense, searchable, human-readable.

    def to_chromadb_format(self) -> Dict:
        """
        Format this episode for storage in ChromaDB.
        Returns the three components ChromaDB needs: id, document, metadata.
        """
        return {
            "id": self.episode_id,
            # Unique document ID in ChromaDB.

            "document": self.episode_summary,
            # The text that gets embedded — the rich prose summary.
            # Semantic search matches against this.

            "metadata": {
                "user_id":       self.user_id,
                # CRITICAL: every query filters by this.
                "session_id":    self.session_id,
                "date":          self.date,
                "timestamp":     self.timestamp,
                "turn_count":    self.turn_count,
                "outcome":       self.outcome,
                "topics":        json.dumps(self.topics),
                # ChromaDB metadata must be str/int/float/bool.
                # Lists must be serialised to JSON strings for storage.
                "decisions_count": len(self.decisions),
                # Store count as int for filtering: "find sessions with decisions".
                "concerns_count":  len(self.concerns),
            },
        }

    def format_for_injection(self, include_numbers: bool = True) -> str:
        """
        Format this episode as a context block for injection into the API call.
        Concise but complete — the model needs the key facts, not the full prose.
        """
        lines = [
            f"PAST SESSION [{self.date}]:",
            f"  Topics     : {', '.join(self.topics)}",
        ]

        if self.decisions:
            lines.append(f"  Decisions  : {' | '.join(self.decisions)}")

        if self.advice_given:
            lines.append(f"  Advice     : {' | '.join(self.advice_given)}")

        if include_numbers and self.key_numbers:
            nums = ', '.join(f"{k}: {v}" for k, v in self.key_numbers.items())
            lines.append(f"  Numbers    : {nums}")

        if self.concerns:
            lines.append(f"  Open issues: {' | '.join(self.concerns)}")

        if self.user_state:
            lines.append(f"  User state : {self.user_state}")

        return "\n".join(lines)


print("Episode dataclass and EpisodeOutcome enum defined.")

Episode dataclass and EpisodeOutcome enum defined.


---
## Part 2 — The Episode Generator

At session end, the full conversation transcript is passed to GPT-4o-mini with a structured extraction prompt. The output is a JSON-formatted episode record.

In [5]:
EPISODE_GENERATION_PROMPT = """You are generating a structured episode record for FinCoach.
An episode is a permanent, immutable record of a financial advisory session.
It will be used for: compliance auditing, case-based reasoning, and user recall.

Analyse the conversation and produce a JSON object with EXACTLY these fields:

{
  "topics": ["list", "of", "main", "topics"],
  "decisions": ["Specific decisions the user committed to — exact and precise"],
  "advice_given": ["Key recommendations FinCoach made — exact and precise"],
  "key_numbers": {"field_name": "value with currency symbol"},
  "user_state": "brief description of user emotional/situational context",
  "concerns": ["Unresolved concerns or questions the user raised"],
  "outcome": "positive | neutral | unresolved | negative",
  "episode_summary": "A rich 3-5 sentence prose summary capturing the full arc of the session."
}

STRICT RULES:
1. Return ONLY valid JSON — no markdown, no explanation.
2. decisions and advice_given must be specific and actionable, not vague.
3. key_numbers must include only figures explicitly stated (salary, FD amounts, SIP targets, etc.).
4. episode_summary must be dense and searchable — it is the text that gets embedded.
5. outcome: 'positive' if user engaged and made decisions; 'neutral' if informational only;
   'unresolved' if concerns remain unanswered; 'negative' if user was frustrated.
6. Never invent facts not in the conversation.
"""
# This prompt is the episodic equivalent of the entity extraction prompt.
# The quality of every downstream recall depends on the quality of this output.


def generate_episode(
    session_id: str,
    user_id: str,
    messages: List[Dict[str, str]],
    session_start_time: Optional[str] = None,
    session_end_time: Optional[str] = None,
) -> Episode:
    """
    Generate a structured episode record from a complete session transcript.

    Args:
        session_id:         The session identifier.
        user_id:            The user identifier.
        messages:           The complete list of messages [{role, content}] from the session.
        session_start_time: ISO timestamp of session start (optional, for duration calc).
        session_end_time:   ISO timestamp of session end.

    Returns:
        A fully populated Episode object.

    Production note:
        This function should be called asynchronously AFTER the session ends.
        The user never waits for this — it runs in the background.
    """

    # Format the transcript for the generation prompt.
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in messages
        if m['role'] in ('user', 'assistant')
        # Skip system messages — they are instructions, not dialogue.
    )

    response = client.chat.completions.create(
        model=EPISODE_MODEL,
        # gpt-4o-mini: structured summarisation task — mini is sufficient.
        max_tokens=600,
        # Episode JSON is typically 300-500 tokens — 600 gives comfortable headroom.
        temperature=0.0,
        # Deterministic — episode records should not vary on re-generation.
        response_format={"type": "json_object"},
        # Force valid JSON output — no parsing overhead.
        messages=[
            {"role": "system", "content": EPISODE_GENERATION_PROMPT},
            {"role": "user",   "content": f"Generate an episode record from this session:\n\n{transcript}"},
        ]
    )

    raw = json.loads(response.choices[0].message.content)
    # Parse the JSON response — guaranteed valid by response_format.

    # Calculate duration if timestamps are available.
    duration = None
    if session_start_time and session_end_time:
        try:
            start = datetime.fromisoformat(session_start_time)
            end   = datetime.fromisoformat(session_end_time)
            duration = round((end - start).total_seconds() / 60, 1)
            # Duration in minutes, rounded to 1 decimal place.
        except Exception:
            duration = None

    now = datetime.now(timezone.utc)

    episode = Episode(
        episode_id       = str(uuid.uuid4()),
        # Fresh UUID4 — unique across all episodes for all users.
        session_id       = session_id,
        user_id          = user_id,
        date             = now.strftime("%Y-%m-%d"),
        # Date only — used for date-based filtering.
        timestamp        = now.isoformat(),
        # Full timestamp — for precise ordering.
        turn_count       = sum(1 for m in messages if m.get('role') == 'assistant'),
        # Count assistant messages = count of complete turns.
        duration_minutes = duration,
        topics           = raw.get("topics", []),
        decisions        = raw.get("decisions", []),
        advice_given     = raw.get("advice_given", []),
        key_numbers      = raw.get("key_numbers", {}),
        user_state       = raw.get("user_state", ""),
        concerns         = raw.get("concerns", []),
        outcome          = raw.get("outcome", EpisodeOutcome.NEUTRAL),
        episode_summary  = raw.get("episode_summary", ""),
    )

    tokens_used = response.usage.total_tokens
    print(f"  [EPISODE] Generated — outcome: {episode.outcome}, "
          f"topics: {episode.topics}, "
          f"decisions: {len(episode.decisions)}, "
          f"tokens: {tokens_used}")

    return episode


print("Episode generation function defined.")

Episode generation function defined.


---
## Part 3 — The Episodic Memory Store

Manages ChromaDB storage and multi-strategy retrieval for episodes.

In [6]:
class EpisodicStore:
    """
    Persistent store for episode records.
    Episodes are embedded by their summary text for semantic retrieval.
    Metadata enables temporal and type-based filtering.
    """

    def __init__(self, user_id: str):
        self.user_id = user_id
        # Scopes all operations to this user.

        self._collection = chroma_client.get_or_create_collection(
            name=EPISODE_COLLECTION,
            metadata={"hnsw:space": "cosine"},
            # Cosine similarity for text embeddings — same as Technique 06.
        )

        print(f"[EpisodicStore] Initialised for user: {self.user_id}")
        print(f"  Existing episodes for this user: {self._count_user_episodes()}")

    def _count_user_episodes(self) -> int:
        """Count episodes belonging to this user."""
        try:
            result = self._collection.get(where={"user_id": {"$eq": self.user_id}})
            return len(result["ids"])
        except Exception:
            return 0

    def save_episode(self, episode: Episode) -> None:
        """
        Embed the episode summary and store in ChromaDB.
        Episodes are immutable — once stored, they are never updated.
        """
        chroma_data = episode.to_chromadb_format()

        # Embed the episode summary for semantic retrieval.
        embedding_response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=episode.episode_summary,
            # The rich prose summary is what gets embedded — not the structured fields.
            # Semantic search matches against this text.
        )
        embedding = embedding_response.data[0].embedding

        self._collection.add(
            ids        =[chroma_data["id"]],
            embeddings =[embedding],
            documents  =[chroma_data["document"]],
            # The episode summary — retrieved alongside the embedding.
            metadatas  =[chroma_data["metadata"]],
            # Structured metadata for filtering.
        )

        print(f"  [STORE] Episode {episode.episode_id[:8]}... saved — "
              f"date: {episode.date}, outcome: {episode.outcome}")

    def retrieve_by_semantic(
        self,
        query: str,
        k: int = 3,
        outcome_filter: Optional[str] = None,
    ) -> List[Dict]:
        """
        Retrieve episodes semantically similar to the query.

        Args:
            query:          The current user message — search anchor.
            k:              Number of episodes to retrieve.
            outcome_filter: Optional filter — e.g. 'positive' for case-based reasoning.

        Returns:
            List of dicts with episode summary, metadata, and distance score.
        """
        total = self._count_user_episodes()
        if total == 0:
            return []

        query_embedding = client.embeddings.create(
            model=EMBEDDING_MODEL, input=query
        ).data[0].embedding
        # Embed the current query for vector similarity search.

        # Build metadata filter.
        where_filter = {"user_id": {"$eq": self.user_id}}
        if outcome_filter:
            where_filter = {
                "$and": [
                    {"user_id": {"$eq": self.user_id}},
                    {"outcome": {"$eq": outcome_filter}},
                ]
            }

        try:
            results = self._collection.query(
                query_embeddings=[query_embedding],
                n_results=min(k, total),
                where=where_filter,
                include=["documents", "metadatas", "distances"],
            )
        except Exception:
            return []

        if not results["documents"] or not results["documents"][0]:
            return []

        return [
            {"summary": doc, "metadata": meta, "distance": round(dist, 4)}
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    def retrieve_recent(
        self,
        n: int = 3,
    ) -> List[Dict]:
        """
        Retrieve the N most recent episodes for this user.
        Used for session continuity — always inject the last few sessions.

        Returns a list sorted newest-first.
        """
        try:
            results = self._collection.get(
                where={"user_id": {"$eq": self.user_id}},
                include=["documents", "metadatas"],
            )
        except Exception:
            return []

        if not results["ids"]:
            return []

        # Sort by timestamp descending — most recent first.
        episodes = sorted(
            zip(results["documents"], results["metadatas"]),
            key=lambda x: x[1].get("timestamp", ""),
            reverse=True,
            # reverse=True: newest timestamp first.
        )

        return [
            {"summary": doc, "metadata": meta}
            for doc, meta in episodes[:n]
        ]

    def retrieve_by_topic(
        self,
        topic_keyword: str,
        k: int = 3,
    ) -> List[Dict]:
        """
        Retrieve episodes where the stored topics JSON contains the keyword.
        ChromaDB supports $contains for string metadata fields.

        Example: retrieve_by_topic('SIP') → all sessions where SIP was discussed.
        """
        try:
            results = self._collection.get(
                where={
                    "$and": [
                        {"user_id": {"$eq": self.user_id}},
                        {"topics":  {"$contains": topic_keyword}},
                        # $contains: checks if the stored topics JSON string
                        # contains this substring.
                    ]
                },
                include=["documents", "metadatas"],
            )
        except Exception:
            return []

        if not results["ids"]:
            return []

        # Sort by timestamp, limit to k.
        sorted_results = sorted(
            zip(results["documents"], results["metadatas"]),
            key=lambda x: x[1].get("timestamp", ""),
            reverse=True,
        )

        return [
            {"summary": doc, "metadata": meta}
            for doc, meta in sorted_results[:k]
        ]

    def get_all_episodes(self) -> List[Dict]:
        """Retrieve all episodes for this user, sorted by date."""
        try:
            results = self._collection.get(
                where={"user_id": {"$eq": self.user_id}},
                include=["documents", "metadatas"],
            )
        except Exception:
            return []

        if not results["ids"]:
            return []

        return sorted(
            [{"summary": d, "metadata": m}
             for d, m in zip(results["documents"], results["metadatas"])],
            key=lambda x: x["metadata"].get("timestamp", ""),
        )


print("EpisodicStore class defined.")

EpisodicStore class defined.


---
## Part 4 — The EpisodicMemory Class

Orchestrates the full episodic memory system: manages the current session, generates episodes at session end, and retrieves relevant past episodes for each new turn.

In [7]:
class EpisodicMemory:
    """
    Full episodic memory system for FinCoach.

    Manages:
    - Current session: recent buffer for conversational flow
    - Episode store: ChromaDB collection of past session records
    - Episode generation: triggered at session close
    - Episode retrieval: semantic + recency strategies per turn

    API context structure:
      [system: persona]
      [system: retrieved past episodes]  ← episodic context
      [recent buffer: current session]   ← conversational flow
    """

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_recent_turns: int = 5,
        # Recent turns to keep verbatim for current session flow.
        episodes_to_inject: int = 2,
        # How many past episodes to inject per turn.
        # 2 is the production default: enough context, not too many tokens.
        retrieval_strategy: str = "hybrid",
        # 'semantic': retrieve by query similarity only
        # 'recent':   retrieve most recent N episodes only
        # 'hybrid':   mix recent + semantic (recommended)
    ):
        self.session_id          = session_id
        self.user_id             = user_id
        self.system_prompt       = system_prompt
        self.max_recent_turns    = max_recent_turns
        self.episodes_to_inject  = episodes_to_inject
        self.retrieval_strategy  = retrieval_strategy

        self.episode_store = EpisodicStore(user_id=user_id)
        # The persistent episode store — all past sessions.

        self._recent_buffer: List[Dict] = []
        # Current session turns verbatim.

        self._full_session_messages: List[Dict] = []
        # Complete record of this session — used for episode generation at close.

        self._total_turns    = 0
        self._session_start  = datetime.now(timezone.utc).isoformat()
        self.created_at      = self._session_start

        print(f"[EpisodicMemory] New session: {self.session_id}")
        print(f"  Past episodes in store : {self.episode_store._count_user_episodes()}")
        print(f"  Retrieval strategy     : {self.retrieval_strategy}")

    def add_message(self, role: str, content: str) -> None:
        """Add a message to the current session buffer and full session log."""
        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'.")

        msg = {"role": role, "content": content}

        self._recent_buffer.append(msg)
        if len(self._recent_buffer) > self.max_recent_turns * 2:
            self._recent_buffer.pop(0)
            # Evict oldest from buffer — keeps current session context bounded.

        self._full_session_messages.append(msg)
        # Full session log — never evicted, used for episode generation.

        if role == "assistant":
            self._total_turns += 1

    def _retrieve_episodes_for_context(
        self,
        current_query: str
    ) -> List[Dict]:
        """
        Retrieve relevant past episodes to inject into the current context.
        Strategy is controlled by self.retrieval_strategy.
        """
        if self.episode_store._count_user_episodes() == 0:
            # No past episodes — first session for this user.
            return []

        if self.retrieval_strategy == "semantic":
            return self.episode_store.retrieve_by_semantic(
                query=current_query,
                k=self.episodes_to_inject,
            )

        elif self.retrieval_strategy == "recent":
            return self.episode_store.retrieve_recent(
                n=self.episodes_to_inject,
            )

        else:  # hybrid — the production default
            # Get 1 most recent + 1 most semantically relevant.
            # Ensures the model always has the last session AND the most relevant one.
            recent    = self.episode_store.retrieve_recent(n=1)
            semantic  = self.episode_store.retrieve_by_semantic(
                query=current_query, k=1
            )

            # Deduplicate by session_id — if recent and semantic return the same episode.
            seen_sessions = set()
            combined = []
            for ep in recent + semantic:
                sid = ep["metadata"].get("session_id", "")
                if sid not in seen_sessions:
                    combined.append(ep)
                    seen_sessions.add(sid)

            return combined[:self.episodes_to_inject]

    def get_messages_for_api(self, current_query: str) -> List[Dict[str, str]]:
        """
        Build the complete API message list with episodic context.

        Structure:
          [system: FinCoach persona]
          [system: past episode records]  ← retrieved episodic memory
          [recent buffer: current session]
        """
        messages = []

        messages.append({"role": "system", "content": self.system_prompt})
        # 1. FinCoach's persona.

        episodes = self._retrieve_episodes_for_context(current_query)
        # 2. Retrieve relevant past episodes.

        if episodes:
            episode_lines = []
            for ep in episodes:
                meta = ep["metadata"]
                ep_date = meta.get("date", "unknown date")
                ep_outcome = meta.get("outcome", "")
                ep_summary = ep["summary"]
                episode_lines.append(
                    f"[Session: {ep_date} | Outcome: {ep_outcome}]\n  {ep_summary}"
                )

            episode_block = (
                "PAST SESSION HISTORY (use for context and continuity):\n"
                + "\n\n".join(episode_lines)
                + "\n\nReference these past sessions to give consistent, personalised advice."
            )
            messages.append({"role": "system", "content": episode_block})
            # Injected as a system message — past context, not current dialogue.

        for msg in self._recent_buffer:
            messages.append(msg)
            # 3. Recent buffer — current session verbatim.

        return messages

    def close_session(self) -> Episode:
        """
        Generate and store an episode record for the current session.
        Call this when the session ends.
        In production: call asynchronously after the last message.

        Returns the generated Episode object.
        """
        session_end = datetime.now(timezone.utc).isoformat()

        print(f"\n[EpisodicMemory] Closing session {self.session_id}...")
        print(f"  Turns: {self._total_turns}, Messages: {len(self._full_session_messages)}")

        episode = generate_episode(
            session_id=self.session_id,
            user_id=self.user_id,
            messages=self._full_session_messages,
            session_start_time=self._session_start,
            session_end_time=session_end,
        )

        self.episode_store.save_episode(episode)
        # Embed and persist to ChromaDB.

        print(f"[EpisodicMemory] Session closed. Episode stored.")
        return episode

    def print_stats(self) -> None:
        """Print episodic memory state."""
        total_eps = self.episode_store._count_user_episodes()
        print(f"\n{'='*58}")
        print(f"  Episodic Memory — Session: {self.session_id}")
        print(f"{'='*58}")
        print(f"  Current session turns  : {self._total_turns}")
        print(f"  Total episodes stored  : {total_eps}")
        print(f"  Retrieval strategy     : {self.retrieval_strategy}")
        print(f"  Episodes injected/turn : {self.episodes_to_inject}")
        print(f"{'='*58}\n")


print("EpisodicMemory class defined.")

EpisodicMemory class defined.


---
## Part 5 — The FinCoach Agent

In [8]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Use past session history provided to give continuous, personalised advice.
- Reference specific decisions and advice from past sessions naturally.
- If a past session shows unresolved concerns, follow up on them.
- Be specific with numbers when they are available.
- Keep responses concise — 3 to 5 sentences unless detail is requested.
- Never recommend specific stocks.
- Always recommend consulting a SEBI-registered advisor for major decisions."""


def chat(
    memory: EpisodicMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """Send one message to FinCoach with episodic memory."""

    memory.add_message(role="user", content=user_message)
    # Add to buffer and full session log.

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(current_query=user_message),
        # Retrieves past episodes and builds context on every call.
    )

    assistant_reply = response.choices[0].message.content
    memory.add_message(role="assistant", content=assistant_reply)

    if verbose:
        print(f"[Turn {memory._total_turns}] "
              f"API input: {response.usage.prompt_tokens} tokens | "
              f"Episodes in store: {memory.episode_store._count_user_episodes()}")

    return assistant_reply


print("FinCoach chat() with episodic memory defined.")

FinCoach chat() with episodic memory defined.


---
## Part 6 — Demo: Session 1 → Episode Generated → Session 2 Recalls It

The complete episodic memory lifecycle: run Session 1, close it (episode generated), start Session 2, watch FinCoach recall Session 1 naturally.

In [9]:
# ── SESSION 1 ──────────────────────────────────────────────────────────────
print("SESSION 1 — Initial FinCoach session")
print("=" * 65)

session1 = EpisodicMemory(
    session_id="session_ep_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
    episodes_to_inject=2,
    retrieval_strategy="hybrid",
)

session1_turns = [
    "Hi, I'm Chiru. My salary is ₹1,20,000 and expenses are ₹60,000. I'm conservative.",
    "I have an FD of ₹50,000 maturing in 3 months. I'm thinking about where to put it.",
    "I never want to invest in equity — only fixed income. My goal is to retire at 55.",
    "OK based on all this, what should I do with my FD maturity amount?",
    # Decision session — FinCoach gives a recommendation. This should be captured in the episode.
]

for i, turn in enumerate(session1_turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    response = chat(session1, turn, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

# Close Session 1 — this generates and stores the episode.
print("\n" + "─" * 65)
print("Closing Session 1 — generating episode...")
episode1 = session1.close_session()

print("\nEpisode generated:")
print(episode1.format_for_injection())

SESSION 1 — Initial FinCoach session
[EpisodicStore] Initialised for user: chiru_001
  Existing episodes for this user: 0
[EpisodicMemory] New session: session_ep_001
  Past episodes in store : 0
  Retrieval strategy     : hybrid

--- Turn 1 ---
User: Hi, I'm Chiru. My salary is ₹1,20,000 and expenses are ₹60,000. I'm conservative.
[Turn 1] API input: 154 tokens | Episodes in store: 0
FinCoach: Hi Chiru! With a salary of ₹1,20,000 and expenses of ₹60,000, you're saving about ₹60,000 each month, which is a great start. Since you're conservative, consider putting a portion of your savings into safer investment options like fixed deposits or recurring deposits, which offer stable returns with low risk. Additionally, you might want to maintain an emergency fund that covers 6-12 months of your expenses. For more personalized investment planning, consulting a SEBI-registered advisor would be beneficial.

--- Turn 2 ---
User: I have an FD of ₹50,000 maturing in 3 months. I'm thinking about wh

In [10]:
# ── SESSION 2 ──────────────────────────────────────────────────────────────
# Completely fresh buffer. Only the episode store persists.
# Chiru does NOT re-introduce himself or repeat his profile.
# FinCoach should recall Session 1 from the episode store.

print("\nSESSION 2 — Returning user, fresh buffer, episode store populated")
print("=" * 65)

session2 = EpisodicMemory(
    session_id="session_ep_002",
    user_id="chiru_001",    # Same user — episode store carries over.
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
    episodes_to_inject=2,
    retrieval_strategy="hybrid",
)

session2_turns = [
    "Hi, I'm back. My FD has matured. Did we discuss what to do with it last time?",
    # RECALL TEST 1: asks explicitly about the past session.
    # Expected: FinCoach recalls the advice given in Session 1.

    "What was the specific amount of the FD and what did you recommend?",
    # RECALL TEST 2: asks for specific numbers from Session 1.
    # Expected: FinCoach recalls ₹50,000 and the advice given.

    "I've also been anxious about whether debt funds are safe given current interest rates.",
    # New concern. Episode for Session 2 should capture this.

    "Given my conservative stance that you should remember, what's your recommendation now?",
    # RECALL TEST 3: references the risk profile from Session 1 without re-stating it.
    # Expected: FinCoach confirms conservative stance and gives updated advice.
]

for i, turn in enumerate(session2_turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    response = chat(session2, turn, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

# Close Session 2.
print("\n" + "─" * 65)
episode2 = session2.close_session()
print("\nSession 2 episode:")
print(episode2.format_for_injection())

session2.print_stats()


SESSION 2 — Returning user, fresh buffer, episode store populated
[EpisodicStore] Initialised for user: chiru_001
  Existing episodes for this user: 1
[EpisodicMemory] New session: session_ep_002
  Past episodes in store : 1
  Retrieval strategy     : hybrid

--- Turn 1 ---
User: Hi, I'm back. My FD has matured. Did we discuss what to do with it last time?
[Turn 1] API input: 296 tokens | Episodes in store: 1
FinCoach: Welcome back, Chiru! Yes, in our last session, we discussed reinvesting your ₹50,000 FD maturity amount into another fixed deposit. We also considered diversifying into a Public Provident Fund (PPF) to balance your conservative investment strategy with long-term growth potential. Have you decided on a specific action, or would you like to explore these or other options further?

--- Turn 2 ---
User: What was the specific amount of the FD and what did you recommend?
[Turn 2] API input: 395 tokens | Episodes in store: 1
FinCoach: In our last session, you mentioned that yo

---
## Part 7 — Querying the Episode Store Directly

In [11]:
# Use the episode store directly to demonstrate all retrieval strategies.
store = EpisodicStore(user_id="chiru_001")

print("EPISODE STORE QUERIES")
print("=" * 65)

# --- Query 1: All episodes ---
print("\n1. All episodes for chiru_001 (chronological):")
all_eps = store.get_all_episodes()
for ep in all_eps:
    meta = ep['metadata']
    print(f"   [{meta['date']}] session: {meta['session_id']} | "
          f"outcome: {meta['outcome']} | "
          f"turns: {meta['turn_count']}")

# --- Query 2: Most recent ---
print("\n2. Most recent episode:")
recent = store.retrieve_recent(n=1)
if recent:
    print(f"   [{recent[0]['metadata']['date']}] {recent[0]['summary'][:120]}...")

# --- Query 3: Semantic search ---
print("\n3. Semantic search: 'what should I do with my FD maturity money?'")
semantic = store.retrieve_by_semantic(
    query="what should I do with my FD maturity money?",
    k=2
)
for ep in semantic:
    print(f"   [{ep['metadata']['date']}] distance={ep['distance']:.4f}: "
          f"{ep['summary'][:100]}...")

# --- Query 4: Compliance audit query ---
print("\n4. Compliance audit: 'find all sessions where decisions were made'")
try:
    audit_results = store._collection.get(
        where={
            "$and": [
                {"user_id": {"$eq": "chiru_001"}},
                {"decisions_count": {"$gt": 0}},
                # $gt: greater than 0 — sessions where at least one decision was made.
            ]
        },
        include=["metadatas", "documents"]
    )
    for meta, doc in zip(audit_results['metadatas'], audit_results['documents']):
        print(f"   [{meta['date']}] session: {meta['session_id']} | "
              f"decisions recorded: {meta['decisions_count']}")
        print(f"   Summary: {doc[:100]}...")
except Exception as e:
    print(f"   Query result: {e}")

[EpisodicStore] Initialised for user: chiru_001
  Existing episodes for this user: 2
EPISODE STORE QUERIES

1. All episodes for chiru_001 (chronological):
   [2026-06-12] session: session_ep_001 | outcome: positive | turns: 4
   [2026-06-12] session: session_ep_002 | outcome: positive | turns: 4

2. Most recent episode:
   [2026-06-12] In this session, Chiru revisited the maturity of a ₹50,000 fixed deposit and confirmed the decision to reinvest it while...

3. Semantic search: 'what should I do with my FD maturity money?'
   [2026-06-12] distance=0.5120: In this session, Chiru revisited the maturity of a ₹50,000 fixed deposit and confirmed the decision ...
   [2026-06-12] distance=0.5688: Chiru, with a salary of ₹1,20,000 and expenses of ₹60,000, is focused on conservative investments an...

4. Compliance audit: 'find all sessions where decisions were made'
   [2026-06-12] session: session_ep_001 | decisions recorded: 2
   Summary: Chiru, with a salary of ₹1,20,000 and expenses of ₹60

---
## Part 8 — Case-Based Reasoning: Learning from Past Episodes

In [12]:
# Case-based reasoning: retrieve successful past episodes to guide current advice.
# This is one of the most powerful applications of episodic memory.

print("CASE-BASED REASONING — Retrieve positive episodes for strategy guidance")
print("=" * 65)

store = EpisodicStore(user_id="chiru_001")

# Retrieve only POSITIVE episodes — where the user was engaged and made decisions.
# These are the 'success cases' we want to learn from.
positive_episodes = store.retrieve_by_semantic(
    query="FD maturity investment fixed income conservative",
    k=3,
    outcome_filter="positive",
    # outcome_filter='positive': only retrieve episodes where the session went well.
    # In case-based reasoning: we want to repeat what worked.
)

print(f"\nFound {len(positive_episodes)} positive past episodes matching this scenario:")
for ep in positive_episodes:
    meta = ep['metadata']
    print(f"\n  Episode: {meta['date']} | outcome: {meta['outcome']}")
    print(f"  Summary: {ep['summary'][:150]}...")

print()
print("Production application:")
print("  When a similar scenario recurs, inject positive past episodes into context.")
print("  The model sees what worked before and tends to apply similar strategies.")
print("  This is case-based reasoning — learning from experience without retraining.")

CASE-BASED REASONING — Retrieve positive episodes for strategy guidance
[EpisodicStore] Initialised for user: chiru_001
  Existing episodes for this user: 2

Found 2 positive past episodes matching this scenario:

  Episode: 2026-06-12 | outcome: positive
  Summary: In this session, Chiru revisited the maturity of a ₹50,000 fixed deposit and confirmed the decision to reinvest it while considering diversification i...

  Episode: 2026-06-12 | outcome: positive
  Summary: Chiru, with a salary of ₹1,20,000 and expenses of ₹60,000, is focused on conservative investments and aims to retire at 55. The session covered strate...

Production application:
  When a similar scenario recurs, inject positive past episodes into context.
  The model sees what worked before and tends to apply similar strategies.
  This is case-based reasoning — learning from experience without retraining.


---
## Part 9 — The Complete Memory Architecture: All Layers Together

In [13]:
print("THE COMPLETE FINCOACH MEMORY ARCHITECTURE — All Techniques Combined")
print("=" * 65)

print("""
EVERY API CALL TO FINCOACH RECEIVES:
──────────────────────────────────────────────────────────────
[system: FinCoach persona]                    ~130 tokens
     +
[system: Entity Profile]          (Technique 07)  ~150 tokens
  User: Chiru | Age: 32 | Salary: ₹1,20,000
  Risk: conservative | Goal: retire at 55
  Constraints: never invest in equity
     +
[system: Past Episodes]           (Technique 09)  ~250 tokens
  [2025-06-12] Discussed FD maturity → recommended debt funds
  [2025-04-10] Started ₹5,000/month SIP in HDFC Short Duration
     +
[system: Retrieved Memories]      (Technique 06)  ~200 tokens
  Semantically relevant past messages from ChromaDB
     +
[recent buffer: current session]  (Technique 05)  ~400 tokens
──────────────────────────────────────────────────────────────
TOTAL:                                            ~1130 tokens


WHAT EACH LAYER ANSWERS:
──────────────────────────
Entity Profile  → "Who is this user right now?" (structured, always current)
Past Episodes   → "What happened in past sessions?" (narrative, timestamped)
Vector Store    → "What did we discuss about X?" (semantic, message-level)
Recent Buffer   → "What are we talking about now?" (verbatim, current session)


WHAT EACH LAYER CANNOT DO:
────────────────────────────
Entity Profile  → No relationships between facts (→ Technique 08 Knowledge Graph)
Past Episodes   → Aggregate view — not message-level (→ Technique 06 for details)
Vector Store    → No temporal logic — stale facts surface (→ Technique 24 Graphiti)
Recent Buffer   → No persistence — lost when session ends


THE STACK SO FAR:
──────────────────
01-05: Short-term (in-session context management)
06: Vector Store (message-level long-term retrieval)
07: Entity Memory (structured, current user profile)
09: Episodic Memory (session-level narrative history)

Coming next:
10: Semantic Memory (generalised facts distilled from episodes)
24: Graphiti (temporal knowledge graph — when facts were true)
""")

THE COMPLETE FINCOACH MEMORY ARCHITECTURE — All Techniques Combined

EVERY API CALL TO FINCOACH RECEIVES:
──────────────────────────────────────────────────────────────
[system: FinCoach persona]                    ~130 tokens
     +
[system: Entity Profile]          (Technique 07)  ~150 tokens
  User: Chiru | Age: 32 | Salary: ₹1,20,000
  Risk: conservative | Goal: retire at 55
  Constraints: never invest in equity
     +
[system: Past Episodes]           (Technique 09)  ~250 tokens
  [2025-06-12] Discussed FD maturity → recommended debt funds
  [2025-04-10] Started ₹5,000/month SIP in HDFC Short Duration
     +
[system: Retrieved Memories]      (Technique 06)  ~200 tokens
  Semantically relevant past messages from ChromaDB
     +
[recent buffer: current session]  (Technique 05)  ~400 tokens
──────────────────────────────────────────────────────────────
TOTAL:                                            ~1130 tokens


WHAT EACH LAYER ANSWERS:
──────────────────────────
Entity Profile  

---
## Key Takeaways

**What you built:** A complete episodic memory system with structured `Episode` dataclass, LLM-powered episode generation using JSON response mode, ChromaDB-backed episode store with semantic + recency + topic retrieval strategies, compliance audit queries, case-based reasoning, and a full two-session lifecycle demo.

---

### The three things to remember

1. **Episodes are immutable.** Once generated and stored, an episode is a permanent record of what the agent knew and said during that session. This is what makes episodic memory an audit trail, not just a convenience feature. In regulated financial services, you cannot rewrite history — episodic memory enforces this at the data model level.

2. **Generate episodes at session close, not turn by turn.** Episode generation is a session-level operation — it captures patterns that individual turns cannot. It runs asynchronously after the session ends so the user never waits. The `close_session()` method is what produces the episode. Never call it mid-session.

3. **Hybrid retrieval (recent + semantic) is the production default.** Recent episodes ensure session continuity — the model always knows what happened last time. Semantic episodes ensure relevance — the model finds episodes about the same topic even if they are old. One of each, deduplicated, gives the best coverage within a fixed token budget.

---

### What Technique 09 adds

Episodic memory stores what happened — specific events, specific sessions, specific dates. 

**Technique 09 — Semantic Memory** extracts the general truths that survive across episodes: not "on April 3rd Chiru was anxious about market volatility" but "Chiru consistently becomes anxious during market volatility and requires reassurance before making decisions." Semantic memory is the distillation of episodic memory into durable, generalised knowledge.

---

### Key Concepts

Episode boundaries: defining where one episode ends and another begins. Boundaries can be session-based, topic-based, or time-based.

Temporal indexing: each episode is tagged with timestamps (start, end). This enables time-based queries.

Episode summaries: short LLM-generated text that captures the gist of an episode. These make retrieval fast.

Semantic retrieval: using embeddings and cosine similarity to find episodes related to the current query.

Episodic vs. semantic memory: episodic memory stores specific experiences. Semantic memory stores generalized knowledge. Both serve different roles.

### When to Use

Multi-session agents where users expect the agent to remember previous interactions.

Project management or coaching agents that need to track progress over time.

Agents that must answer "when did we..." or "what happened during..." style questions.

Scenarios where the temporal sequence of events matters for reasoning.

### Limitations

Episode boundary detection can produce segments that are too fine or too coarse.

Summaries lose fine-grained detail. Specific numbers or facts may be lost.

Retrieval adds latency (embedding call plus episode scan on every turn).

Storage grows with every session. Long-lived agents need pruning or archival strategies.